# Multimodal Heart Disease Prediction — Research Pipeline

**Investigational & Research Prototype Only.** Not for clinical diagnosis or triage.

This notebook demonstrates the end-to-end multimodal deep learning pipeline combining:
1. **12-Lead ECG Waveforms** (1D Multi-Scale CNN + BiLSTM + Attention Pooling)
2. **Structured Clinical Features** (Missingness-aware MLP)
3. **Dynamic Cross-Modal Fusion** (Gated Fusion & Cross-Attention)
4. **Patient-Grouped Leakage-Free Splitting**
5. **Explainability & Attribution Analysis** (Integrated Gradients & Tabular Attributions)
6. **Versioned Model Packaging**

In [ ]:
import os, sys, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Ensure project root is in path
sys.path.append(os.path.abspath(".."))

from src.data.synthetic_generator import generate_synthetic_multimodal_cohort
from src.data.contracts import DEFAULT_CLINICAL_FEATURES
from src.data.ecg_processor import ECGProcessor
from src.data.clinical_processor import ClinicalProcessor
from src.data.dataset import MultimodalDataset, patient_level_split, create_dataloaders
from src.models.multimodal_classifier import MultimodalClassifier
from src.models.baselines import ECGOnlyClassifier, ClinicalOnlyClassifier
from src.models.trainer import MultimodalTrainer, compute_binary_metrics, find_optimal_threshold
from src.models.explainability import compute_ecg_saliency, compute_clinical_attributions
from src.models.artifact import ModelArtifactManager

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Computation Device:", DEVICE)

## 1. Cohort Ingestion & Signal Synthesis

We generate a cohort of 300 patients with correlated clinical risk scores and 12-lead ECG waveforms (500 Hz, 5.0 seconds = 2,500 samples per lead).

In [ ]:
df_clinical, raw_ecgs = generate_synthetic_multimodal_cohort(num_patients=300, seed=SEED)
print(f"Cohort Size: {len(df_clinical)} records from {df_clinical['patient_id'].nunique()} patients")
print("ECG Tensor Shape:", raw_ecgs.shape) # [N, 12, 2500]
df_clinical.head()

## 2. Patient-Level Leakage-Free Splitting

Crucial clinical ML principle: **Never split serial records or beats from the same patient across train and test!**
We use `GroupShuffleSplit` on `patient_id` to ensure zero patient leakage.

In [ ]:
train_idx, val_idx, test_idx = patient_level_split(
    df_clinical,
    patient_col="patient_id",
    target_col="target",
    val_size=0.15,
    test_size=0.15,
    seed=SEED
)
print(f"Partition Counts -> Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

## 3. Preprocessing: Fit on Training Split Only

- Butterworth bandpass filtering (0.5 Hz – 45.0 Hz) and 50 Hz notch filtering for ECG
- Median imputation and z-score scaling for clinical variables, fitted exclusively on training data

In [ ]:
ecg_proc = ECGProcessor(target_fs=500.0, nominal_samples=2500)
proc_ecgs = np.zeros_like(raw_ecgs)
for i in range(len(raw_ecgs)):
    proc_ecgs[i] = ecg_proc.process(raw_ecgs[i], orig_fs=500.0)

feature_cols = [c for c in DEFAULT_CLINICAL_FEATURES if c in df_clinical.columns]
clin_proc = ClinicalProcessor(feature_names=feature_cols)
clin_proc.fit(df_clinical.iloc[train_idx][feature_cols])

X_train_c = clin_proc.transform(df_clinical.iloc[train_idx][feature_cols])
X_val_c = clin_proc.transform(df_clinical.iloc[val_idx][feature_cols])
X_test_c = clin_proc.transform(df_clinical.iloc[test_idx][feature_cols])

y_train = df_clinical.iloc[train_idx]["target"].to_numpy()
y_val = df_clinical.iloc[val_idx]["target"].to_numpy()
y_test = df_clinical.iloc[test_idx]["target"].to_numpy()

train_ds = MultimodalDataset(proc_ecgs[train_idx], X_train_c, y_train)
val_ds = MultimodalDataset(proc_ecgs[val_idx], X_val_c, y_val)
test_ds = MultimodalDataset(proc_ecgs[test_idx], X_test_c, y_test)

loaders = create_dataloaders(train_ds, val_ds, test_ds, batch_size=32)

## 4. Train Multimodal Model (Gated Fusion)

In [ ]:
model = MultimodalClassifier(
    in_leads=12,
    n_clinical_features=len(feature_cols),
    embedding_dim=128,
    fusion_type="gated"
)
trainer = MultimodalTrainer(model, patience=7)
results = trainer.fit(loaders["train"], loaders["val"], epochs=15, verbose=True)

## 5. Locked Test Evaluation & Metrics

In [ ]:
val_loss, y_val_true, y_val_probs = trainer.evaluate(loaders["val"])
opt_thresh = find_optimal_threshold(y_val_true, y_val_probs)
print(f"Tuned Decision Threshold: {opt_thresh:.3f}")

test_loss, y_test_true, y_test_probs = trainer.evaluate(loaders["test"])
test_metrics = compute_binary_metrics(y_test_true, y_test_probs, threshold=opt_thresh)
for k, v in test_metrics.items():
    print(f"{k:<20}: {v}")

## 6. Explainability: ECG Saliency & Clinical Attributions

In [ ]:
sample_ecg = torch.from_numpy(proc_ecgs[test_idx[0]]).float()
sample_clin = torch.from_numpy(X_test_c[0]).float()

saliency = compute_ecg_saliency(model, sample_ecg, sample_clin, steps=10)
print("Top Saliency Leads:")
for lead in saliency["lead_ranking"][:3]:
    print(f"  - {lead['lead']}: weight={lead['weight']:.4f}")

attributions = compute_clinical_attributions(model, sample_ecg, sample_clin, feature_cols)
print("\nTop Clinical Attributions:")
for attr in attributions[:4]:
    print(f"  - {attr['feature']}: score={attr['attribution_score']:.4f} ({attr['direction']})")

## 7. Model Artifact Packaging & Integrity Hash

In [ ]:
artifact_path = "../artifacts/best_multimodal_model.pt"
sha256 = ModelArtifactManager.save_artifact(
    model=model,
    clinical_processor=clin_proc,
    save_path=artifact_path,
    threshold=opt_thresh,
    version="v1.0.0-multimodal-gated",
    metrics=test_metrics
)
print(f"Model artifact successfully exported to {artifact_path}")
print(f"SHA-256 Checksum: {sha256}")